In [1]:
import pandas as pd
import numpy as np

deliveries = pd.read_csv('data/deliveries.csv')
print(deliveries.columns.tolist())

['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball', 'batter', 'bowler', 'non_striker', 'batsman_runs', 'extra_runs', 'total_runs', 'extras_type', 'is_wicket', 'player_dismissed', 'dismissal_kind', 'fielder']


In [2]:
# Filter inning 2 only
inning2 = deliveries[deliveries['inning'] == 2].copy()
print(inning2.shape)

(125741, 17)


In [3]:
## Engineer In-Match Features
inning2['runs_scored'] = inning2.groupby('match_id')['total_runs'].cumsum()
inning2['wickets_fallen'] = inning2.groupby('match_id')['is_wicket'].cumsum()
inning2['wickets_remaining'] = 10 - inning2['wickets_fallen']
inning2['balls_bowled'] = inning2.groupby('match_id').cumcount() + 1
inning2['overs_bowled'] = inning2['balls_bowled'] / 6

print(inning2[['match_id', 'over', 'ball', 'runs_scored', 'wickets_remaining', 'overs_bowled']].head(10))

     match_id  over  ball  runs_scored  wickets_remaining  overs_bowled
124    335982     0     1            1                 10      0.166667
125    335982     0     2            2                 10      0.333333
126    335982     0     3            2                 10      0.500000
127    335982     0     4            3                 10      0.666667
128    335982     0     5            4                 10      0.833333
129    335982     0     6            4                 10      1.000000
130    335982     0     7            4                 10      1.166667
131    335982     1     1            4                  9      1.333333
132    335982     1     2            4                  9      1.500000
133    335982     1     3            8                  9      1.666667


In [4]:
## Calculate Target from Inning 1
inning1_totals = deliveries[deliveries['inning'] == 1].groupby('match_id')['total_runs'].sum().reset_index()
inning1_totals.columns = ['match_id', 'target']
inning1_totals['target'] = inning1_totals['target'] + 1  # target is always +1 in cricket

inning2 = inning2.merge(inning1_totals, on='match_id', how='left')

# Now calculate runs remaining and required run rate
inning2['runs_remaining'] = inning2['target'] - inning2['runs_scored']
inning2['balls_remaining'] = 120 - inning2['balls_bowled']
inning2['overs_remaining'] = inning2['balls_remaining'] / 6
inning2['crr'] = inning2['runs_scored'] / inning2['overs_bowled']  # current run rate
inning2['rrr'] = inning2['runs_remaining'] / inning2['overs_remaining']  # required run rate

print(inning2[['match_id', 'runs_scored', 'target', 'runs_remaining', 'crr', 'rrr']].head(10))

   match_id  runs_scored  target  runs_remaining       crr        rrr
0    335982            1     223             222  6.000000  11.193277
1    335982            2     223             221  6.000000  11.237288
2    335982            2     223             221  4.000000  11.333333
3    335982            3     223             220  4.500000  11.379310
4    335982            4     223             219  4.800000  11.426087
5    335982            4     223             219  4.000000  11.526316
6    335982            4     223             219  3.428571  11.628319
7    335982            4     223             219  3.000000  11.732143
8    335982            4     223             219  2.666667  11.837838
9    335982            8     223             215  4.800000  11.727273


In [5]:
## Merge with Matches to get Winner
matches = pd.read_csv('data/matches.csv')

# Keep only match_id and winner from matches
match_winners = matches[['id', 'winner']].rename(columns={'id': 'match_id'})

# Merge
inning2 = inning2.merge(match_winners, on='match_id', how='left')

# Drop rows where winner is null (no result matches)
inning2 = inning2.dropna(subset=['winner'])

print(inning2.shape)
print(inning2[['match_id', 'batting_team', 'bowling_team', 'runs_scored', 'target', 'winner']].head())

(125714, 29)
   match_id                 batting_team           bowling_team  runs_scored  \
0    335982  Royal Challengers Bangalore  Kolkata Knight Riders            1   
1    335982  Royal Challengers Bangalore  Kolkata Knight Riders            2   
2    335982  Royal Challengers Bangalore  Kolkata Knight Riders            2   
3    335982  Royal Challengers Bangalore  Kolkata Knight Riders            3   
4    335982  Royal Challengers Bangalore  Kolkata Knight Riders            4   

   target                 winner  
0     223  Kolkata Knight Riders  
1     223  Kolkata Knight Riders  
2     223  Kolkata Knight Riders  
3     223  Kolkata Knight Riders  
4     223  Kolkata Knight Riders  


In [6]:
## Clean Team Names
team_name_map = {
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings',
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru',
    'Rising Pune Supergiant': 'Rising Pune Supergiants'
}

inning2['batting_team'] = inning2['batting_team'].replace(team_name_map)
inning2['bowling_team'] = inning2['bowling_team'].replace(team_name_map)
inning2['winner'] = inning2['winner'].replace(team_name_map)

print(inning2['batting_team'].unique())

['Royal Challengers Bengaluru' 'Punjab Kings' 'Delhi Capitals'
 'Kolkata Knight Riders' 'Rajasthan Royals' 'Mumbai Indians'
 'Chennai Super Kings' 'Deccan Chargers' 'Pune Warriors'
 'Kochi Tuskers Kerala' 'Sunrisers Hyderabad' 'Rising Pune Supergiants'
 'Gujarat Lions' 'Gujarat Titans' 'Lucknow Super Giants']


In [7]:
## Create Target Variable
inning2['batting_team_won'] = (inning2['batting_team'] == inning2['winner']).astype(int)

print(inning2['batting_team_won'].value_counts())

batting_team_won
1    65346
0    60368
Name: count, dtype: int64


In [11]:
## Train Model on In-Match Features
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Define features and target
features = ['runs_scored', 'wickets_remaining', 'overs_bowled', 
            'runs_remaining', 'crr', 'rrr', 'target']

# Replace infinite values and nulls caused by division by zero
X = inning2[features].replace([np.inf, -np.inf], 0).fillna(0)
y = inning2['batting_team_won']

# Train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Random Forest
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluate
predictions = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, predictions))

Accuracy: 0.888279043869069
